# AIND Ephys Dispatch

Build an `AINDEPhysDispatchScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysDispatchTask` for each coordinate.

The dispatch task wraps the [aind-ephys-job-dispatch](https://github.com/AllenNeuralDynamics/aind-ephys-job-dispatch/blob/main/code/run_capsule.py) preprocessing CLI. Each `execute()` prints the command line that would be invoked for that coordinate.

This notebook targets the toy SpikeInterface recording produced by `generate_toy_recording.ipynb` (saved at `./toy_example_recording`). It is a `BinaryFolderRecording`, so we use the `spikeinterface` input mode with `reader_type='binaryfolder'`.

## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.spike_sorting.dispatch.blocks import (
    DispatchBasic,
    DispatchDataDependent,
    DispatchDebug,
    SpikeInterfaceInfo,
)

## Build the scan config

Any field set to a list becomes a dimension of the parameter scan. Below we sweep `min_recording_duration` (2 values) and `debug_mode` (2 values), giving a 2x2 = 4 coordinate scan.

The recording is loaded by SpikeInterface via the `binaryfolder` extractor, pointing at the saved toy recording. The dispatch CLI runs the script from `code/run`, so we resolve an absolute path here to make the printed command directly invokable.

In [ ]:
recording_folder = (Path.cwd() / "toy_example_recording").resolve()
assert recording_folder.exists(), (
    f"{recording_folder} not found. Run generate_toy_recording.ipynb first."
)

scan_config = obi.AINDEPhysDispatchScanConfig(
    dispatch_basic=DispatchBasic(
        split_segments=True,
        split_groups=True,
        skip_timestamps_check=False,
    ),
    dispatch_data_dependent=DispatchDataDependent(
        input_format="spikeinterface",
        multi_session_data=False,
        spikeinterface_info=SpikeInterfaceInfo(
            reader_type="binaryfolder",
            reader_kwargs={"folder_path": str(recording_folder)},
        ),
    ),
    dispatch_debug=DispatchDebug(
        debug_mode=[True, False],
        debug_duration=10.0,
    ),
)

## Generate the grid scan and run the task for each coordinate

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/aind_ephys_dispatch/grid_scan",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

## Inspect the generated single configs

In [ ]:
for single_config in grid_scan.single_configs:
    print("---")
    print("min_recording_duration:", single_config.dispatch_basic.min_recording_duration)
    print("debug_mode:            ", single_config.dispatch_debug.debug_mode)
    print("command:")
    print("  ", single_config.command_line_representation())